An important aspect in deep-learning-based identification (and in any deep-learning applications, in general) is the initialization of the parameters. To avoid converging to a local minima, training should be started from multiple independent initial parameters (according to a given initialization scheme). However, independently training models from these starting points can be time-consuming, thus, we can utilize parallel training, then we can select the best-performing structure based on a validation data set (can be the testing or training set, etc.).

Let's start with te previous example problem:

In [ ]:
# basic imports and setting JAX options
import numpy as np
import jax
from model_augmentation_jax import LinearTimeInvariantSystem, StaticLFRAugmentation
from model_augmentation_jax.utils import compute_normalization_constants, find_best_model
import time


jax.config.update('jax_platform_name', 'cpu')
if not jax.config.jax_enable_x64:
    jax.config.update("jax_enable_x64", True)  # Enable 64-bit computations

# Generate or load data
np.random.seed(0)
U = np.random.normal(size=10_000) # Input sequence
x = [0, 0] # Initial state
ylist = [] # Output sequence
for uk in U:
    ylist.append(x[0] + np.random.normal(loc=0., scale=0.01))  # Compute output
    x = 0.9 * x[0] + 0.1 * x[1] + 0.1 * uk + 0.02 * x[0] * x[1], \
       -0.2 * x[0] + 0.95 * x[1] + 0.05 * uk - 0.1 * x[0]**3 # Advance state

# Split dataset
Y = np.array(ylist)
Y_train = Y[:9000]
Y_test = Y[9000:]
U_train = U[:9000]
U_test = U[9000:]

# create LTI baseline model (with approximate params)
A_mx = np.array([[0.88, 0.11], [-0.2, 0.94]])
B_mx = np.array([[0.1], [0.]])
C_mx = np.array([[1., 0.]])
fp_model = LinearTimeInvariantSystem(A=A_mx, B=B_mx, C=C_mx)

# simulate baseline model to approximate constants for normalization
Yhat_train_base, Xhat_train_base = fp_model.simulate(U_train)  # starts from x0 = 0
norm = compute_normalization_constants(U_train, Y_train, Xhat_train_base)

Normally, when training a single model, an initial seed can be provided for parameter initialization by the `seed` argument.

In [2]:
# create augmented model
model = StaticLFRAugmentation(known_sys=fp_model, hidden_layers=2, nodes_per_layer=8, activation="tanh", nz=2, nw=2, norm_dict=norm, seed=42)

Alternatively, we can provide a list with multiple different seeds:

In [3]:
seeds = [0, 13, 42, 123]  # alternative example: seeds = range(8)
model = StaticLFRAugmentation(known_sys=fp_model, hidden_layers=2, nodes_per_layer=8, activation="tanh", nz=2, nw=2, norm_dict=norm, seed=seeds)

Optimization and regularization options can be chosen tha same as previously. For training, the `fit_parallel` method should be called. note that it now returns a list of all trained models. This function requires the list of initial seeds, as well as an optional `n_jobs` argument that specifies the numebr of parallel threads to be appleid.

In [ ]:
# set training options
model.set_optimization_parameters(adam_epochs=100, lbfgs_epochs=500, train_x0=True, verbosity=50)

# can also set regularization if necessary
# model.set_regularization_terms()

start_time = time.time()
models = model.fit_parallel(Y_train, U_train, seeds, 4)
end_time = time.time()

In [5]:
print(f"All {len(seeds)} models trained in parallel in : {end_time - start_time} seconds.")

All 4 models trained in parallel in : 5.991178035736084 seconds.


The best model can be found by using the `find_best_model`, which computes the RMSE on the provided input-output data and finds the model with the lowest error. Here, we have cheated a little bit and utilized the test data for this reason. Naturally the same can be applied using the training data set or an alternate validation set which is independent of both the training and test data.

In [6]:
best_model, best_rmse = find_best_model(models, Y_test, U_test, seeds=seeds, state_estim_len=10)

Evaluating models...



An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.


Scores:
Model 0: RMSE = 0.013324400275538656
Model 1: RMSE = 0.01254638651017376
Model 2: RMSE = 0.01129948143530567
Model 3: RMSE = 0.013446172441687554
Best model: 2, score: 0.01129948143530567 at seed 42
